In [1]:
from google.colab import userdata


In [2]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
changed 22 packages in 2s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴

In [3]:
!pip install streamlit anthropic

In [4]:
%%writefile app_manglar.py
# -*- coding: utf-8 -*-
"""
MANGLEE M4 - Chatbot con datos en vivo desde Google Earth Engine
Ejecutar desde Google Colab.

INSTRUCCIONES EN COLAB:
=======================
Celda 1:
  !pip install streamlit anthropic pyngrok geemap earthengine-api

Celda 2:
  import ee
  ee.Authenticate()
  (esto guarda las credenciales en ~/.config/earthengine/)

Celda 3:
  %%writefile app_manglar.py
  [pegar todo este archivo]

Celda 4:
  !streamlit run app_manglar.py --server.port 8501 &

Celda 5 (opcion A - localtunnel):
  !npm install -g localtunnel
  !lt --port 8501

Celda 5 (opcion B - ngrok):
  from pyngrok import ngrok
  public_url = ngrok.connect(8501)
  print(f'Abre: {public_url}')
"""

import streamlit as st
from anthropic import Anthropic
import ee
import json

# ======================== CONFIGURACION ========================

ANTHROPIC_API_KEY = "AQUI_TU_API_KEY"  # Reemplazar

GEE_PROJECT = 'gee-space-hack'
ASSET_ROOT = 'projects/gee-space-hack/assets/SpaceHack'

FLOOD_SCENARIOS = {
    'Marea normal': 3.0,
    'Marea alta': 4.0,
    'Extremo: marea+lluvia': 5.0,
    'Catastrofico': 6.0
}

ATTENUATION_CM_PER_100M = 30
MAX_ATTENUATION_M = 1.5
COST_PER_HA_USD = 50000

# ======================== CONEXION A GEE ========================

@st.cache_resource
def init_gee():
    try:
        ee.Initialize(project=GEE_PROJECT)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=GEE_PROJECT)
    return True

@st.cache_data(ttl=3600)
def load_all_data():
    """Carga y calcula todos los datos desde GEE. Se ejecuta una vez y se cachea."""

    results = {
        'mangrove_areas': {},
        'flood_scenarios': [],
        'restoration': {},
        'change_rates': [],
    }

    # --- AOI ---
    AOI = ee.FeatureCollection(f'{ASSET_ROOT}/AOI_GreaterGuayaquil_v2')
    aoi_geom = AOI.geometry()

    # --- Manglares SERVIR ---
    man_assets = {
        2018: f'{ASSET_ROOT}/data/MAN_2018',
        2020: f'{ASSET_ROOT}/data/MAN_2020',
        2022: f'{ASSET_ROOT}/data/MAN_2022',
    }

    mangrove_rasters = {}
    for year, path in man_assets.items():
        fc = ee.FeatureCollection(path)
        raster = ee.Image(0).paint(fc, 1).rename('manglar').clip(aoi_geom)
        mangrove_rasters[year] = raster

        area = (raster.multiply(ee.Image.pixelArea()).divide(10000)
            .reduceRegion(ee.Reducer.sum(), aoi_geom, 10, maxPixels=1e10)
            .get('manglar').getInfo())
        results['mangrove_areas'][year] = round(area or 0, 1)

    # --- Clasificaciones propias (si existen) ---
    for year in [2023, 2024, 2025]:
        try:
            img = ee.Image(f'{ASSET_ROOT}/MANGROVE_{year}')
            area = (img.multiply(ee.Image.pixelArea()).divide(10000)
                .reduceRegion(ee.Reducer.sum(), aoi_geom, 10, maxPixels=1e10)
                .get('manglar').getInfo())
            results['mangrove_areas'][year] = round(area or 0, 1)
            mangrove_rasters[year] = img
        except Exception:
            pass

    # --- Tasas de cambio ---
    years_sorted = sorted(results['mangrove_areas'].keys())
    for i in range(1, len(years_sorted)):
        y0, y1 = years_sorted[i-1], years_sorted[i]
        a0 = results['mangrove_areas'][y0]
        a1 = results['mangrove_areas'][y1]
        diff = a1 - a0
        pct = (diff / a0) * 100 if a0 > 0 else 0
        interval = y1 - y0
        annual = pct / interval if interval > 0 else 0
        results['change_rates'].append({
            'periodo': f'{y0}-{y1}',
            'cambio_ha': round(diff, 1),
            'cambio_pct': round(pct, 2),
            'tasa_anual_pct': round(annual, 2)
        })

    # --- DEM y capas auxiliares ---
    dem = ee.Image('USGS/SRTMGL1_003').select('elevation').clip(aoi_geom)

    latest_year = max(mangrove_rasters.keys())
    mangrove_current = mangrove_rasters[latest_year]

    worldpop = (ee.ImageCollection('WorldPop/GP/100m/pop/ECU')
        .filterDate('2020-01-01', '2020-12-31')
        .mosaic().clip(aoi_geom).rename('population'))

    ghsl = ee.Image('JRC/GHSL/P2023A/GHS_BUILT_S/2020').clip(aoi_geom)
    urban_mask = ghsl.select('built_surface').gt(0).rename('urban')

    # --- Ancho de manglar y atenuacion ---
    mangrove_width = mangrove_current.fastDistanceTransform(256).sqrt().multiply(10)
    attenuation = (mangrove_width.divide(100)
        .multiply(ATTENUATION_CM_PER_100M / 100)
        .min(MAX_ATTENUATION_M))

    # --- Modelo de inundacion ---
    for name, level in FLOOD_SCENARIOS.items():
        effective = ee.Image.constant(level).subtract(attenuation)
        flooded_with = dem.lt(effective).And(dem.lt(level))
        flooded_without = dem.lt(level)

        area_with = (flooded_with.multiply(ee.Image.pixelArea()).divide(10000)
            .reduceRegion(ee.Reducer.sum(), aoi_geom, 30, maxPixels=1e10)
            .get('elevation').getInfo())

        area_without = (flooded_without.multiply(ee.Image.pixelArea()).divide(10000)
            .reduceRegion(ee.Reducer.sum(), aoi_geom, 30, maxPixels=1e10)
            .get('elevation').getInfo())

        pop_with = (flooded_with.multiply(worldpop)
            .reduceRegion(ee.Reducer.sum(), aoi_geom, 100, maxPixels=1e10)
            .get('population').getInfo())

        pop_without = (flooded_without.multiply(worldpop)
            .reduceRegion(ee.Reducer.sum(), aoi_geom, 100, maxPixels=1e10)
            .get('population').getInfo())

        urban_with = (flooded_with.And(urban_mask)
            .multiply(ee.Image.pixelArea()).divide(10000)
            .reduceRegion(ee.Reducer.sum(), aoi_geom, 30, maxPixels=1e10)
            .values().get(0).getInfo())

        urban_without = (flooded_without.And(urban_mask)
            .multiply(ee.Image.pixelArea()).divide(10000)
            .reduceRegion(ee.Reducer.sum(), aoi_geom, 30, maxPixels=1e10)
            .values().get(0).getInfo())

        area_protected = (area_without or 0) - (area_with or 0)
        pop_protected = int(pop_without or 0) - int(pop_with or 0)
        urban_protected = (urban_without or 0) - (urban_with or 0)
        cost_avoided = urban_protected * COST_PER_HA_USD

        results['flood_scenarios'].append({
            'escenario': name,
            'nivel_m': level,
            'area_inundada_con_manglar_ha': round(area_with or 0, 1),
            'area_inundada_sin_manglar_ha': round(area_without or 0, 1),
            'area_protegida_ha': round(area_protected, 1),
            'poblacion_expuesta_con_manglar': int(pop_with or 0),
            'poblacion_expuesta_sin_manglar': int(pop_without or 0),
            'poblacion_protegida': pop_protected,
            'area_urbana_protegida_ha': round(urban_protected, 1),
            'danos_evitados_usd': round(cost_avoided, 0)
        })

    # --- Restauracion ---
    man2018_raster = mangrove_rasters[2018]
    lost = man2018_raster.And(mangrove_current.Not())
    restorable = lost.And(urban_mask.Not()).And(dem.lt(10))

    area_restorable = (restorable.multiply(ee.Image.pixelArea()).divide(10000)
        .reduceRegion(ee.Reducer.sum(), aoi_geom, 10, maxPixels=1e10)
        .values().get(0).getInfo())

    results['restoration'] = {
        'area_restaurable_ha': round(area_restorable or 0, 1),
        'criterios': 'Manglar en 2018, no presente en ano mas reciente, no urbanizado, elevacion < 10m',
        'costo_estimado_por_ha_usd': '3,000-8,000',
    }

    results['metadata'] = {
        'manglar_referencia': latest_year,
        'fuentes': 'Sentinel-1/2, SERVIR-Amazonia, SRTM, WorldPop, GHSL, WorldCover',
        'modelo': 'Bathtub con atenuacion por manglar (30cm/100m)',
    }

    return results

# ======================== CONSTRUIR CONTEXTO PARA LLM ========================

def build_context(data):
    ctx = "DATOS EN VIVO DESDE GOOGLE EARTH ENGINE\n"
    ctx += "=" * 50 + "\n\n"

    ctx += "COBERTURA DE MANGLAR (hectareas):\n"
    for year in sorted(data['mangrove_areas'].keys()):
        source = "SERVIR-Amazonia" if year <= 2022 else "Clasificacion RF propia"
        ctx += f"  {year}: {data['mangrove_areas'][year]:,.1f} ha ({source})\n"

    ctx += "\nTASAS DE CAMBIO:\n"
    for c in data['change_rates']:
        sign = '+' if c['cambio_ha'] > 0 else ''
        ctx += f"  {c['periodo']}: {sign}{c['cambio_ha']:,.1f} ha ({sign}{c['cambio_pct']}%, {sign}{c['tasa_anual_pct']}%/ano)\n"

    ctx += "\nMODELO DE INUNDACION:\n"
    for s in data['flood_scenarios']:
        ctx += f"\n  Escenario: {s['escenario']} ({s['nivel_m']}m)\n"
        ctx += f"    Area inundada CON manglar: {s['area_inundada_con_manglar_ha']:,.1f} ha\n"
        ctx += f"    Area inundada SIN manglar: {s['area_inundada_sin_manglar_ha']:,.1f} ha\n"
        ctx += f"    Area protegida por manglar: {s['area_protegida_ha']:,.1f} ha\n"
        ctx += f"    Poblacion expuesta CON manglar: {s['poblacion_expuesta_con_manglar']:,}\n"
        ctx += f"    Poblacion expuesta SIN manglar: {s['poblacion_expuesta_sin_manglar']:,}\n"
        ctx += f"    Poblacion protegida: {s['poblacion_protegida']:,}\n"
        ctx += f"    Danos evitados: ${s['danos_evitados_usd']:,.0f} USD\n"

    ctx += f"\nRESTAURACION:\n"
    ctx += f"  Area restaurable: {data['restoration']['area_restaurable_ha']:,.1f} ha\n"
    ctx += f"  Criterios: {data['restoration']['criterios']}\n"
    ctx += f"  Costo estimado: ${data['restoration']['costo_estimado_por_ha_usd']} USD/ha\n"

    ctx += f"\nMETADATOS:\n"
    ctx += f"  Manglar de referencia: {data['metadata']['manglar_referencia']}\n"
    ctx += f"  Fuentes: {data['metadata']['fuentes']}\n"
    ctx += f"  Modelo: {data['metadata']['modelo']}\n"

    ctx += "\nCONTEXTO ADICIONAL:\n"
    ctx += "  Greater Guayaquil: Guayaquil, Duran, Daule, Samborondon (~3.3M hab)\n"
    ctx += "  Guayaquil concentra ~25% del PIB de Ecuador\n"
    ctx += "  Mareas del rio Guayas: hasta 5 metros\n"
    ctx += "  Lluvias extremas 2023-2025: >70mm en un dia\n"
    ctx += "  Perdida historica de manglar 1970-2000: ~50%\n"
    ctx += "  Stakeholders: GAD Guayaquil, MAATE, SENAGUA, SGR, INOCAR, comunidades costeras\n"
    ctx += "  Marco legal: Codigo Organico del Ambiente Art.99-103, custodias de manglar\n"

    return ctx

# ======================== INTERFAZ ========================

st.set_page_config(page_title="ManglarGYE", page_icon="🌿", layout="centered")

st.title("ManglarGYE")
st.caption("Asistente con datos en vivo desde Google Earth Engine")

# Inicializar GEE
with st.spinner("Conectando a Google Earth Engine..."):
    init_gee()

# Cargar datos
with st.spinner("Calculando estadisticas desde GEE (esto toma ~2 minutos la primera vez)..."):
    data = load_all_data()

st.success("Datos cargados desde GEE")

# Mostrar resumen en sidebar
with st.sidebar:
    st.header("Datos en vivo")
    st.subheader("Manglar por ano")
    for year in sorted(data['mangrove_areas'].keys()):
        st.metric(f"{year}", f"{data['mangrove_areas'][year]:,.0f} ha")

    st.subheader("Proteccion (escenario 5m)")
    extreme = next((s for s in data['flood_scenarios'] if s['nivel_m'] == 5.0), None)
    if extreme:
        st.metric("Area protegida", f"{extreme['area_protegida_ha']:,.0f} ha")
        st.metric("Poblacion protegida", f"{extreme['poblacion_protegida']:,}")
        st.metric("Danos evitados", f"${extreme['danos_evitados_usd']:,.0f}")

    st.subheader("Restauracion")
    st.metric("Area restaurable", f"{data['restoration']['area_restaurable_ha']:,.0f} ha")

    st.markdown("---")
    st.subheader("Preguntas sugeridas")
    suggestions = [
        "Cuanta area de manglar hay actualmente?",
        "Que pasa si la marea sube a 5 metros?",
        "Cuantas personas protegen los manglares?",
        "Donde se puede restaurar manglar?",
        "Cuanto dinero se pierde sin manglares?",
        "Compara el escenario de 3m con el de 6m",
        "Que recomendarias al alcalde de Guayaquil?",
        "Cuales son las limitaciones del estudio?",
    ]
    for q in suggestions:
        if st.button(q, key=q):
            st.session_state.pending_question = q

# Sistema del LLM
context = build_context(data)

SYSTEM_PROMPT = f"""Eres ManglarGYE, un asistente experto en manglares y riesgo de inundacion
para Greater Guayaquil, Ecuador. Proyecto SpaceHACK 2026.

Tienes acceso a datos calculados en tiempo real desde Google Earth Engine:

{context}

Reglas:
- Responde en espanol
- Usa SOLO los datos proporcionados, no inventes cifras
- Se conciso pero informativo
- Contextualiza cifras grandes (equivalencias, comparaciones)
- Si te preguntan algo fuera del alcance, dilo
- Puedes hacer recomendaciones basadas en los datos
- Si comparan escenarios, presenta los datos lado a lado
- Explica de forma simple para audiencias no tecnicas
"""

# Chat
if "messages" not in st.session_state:
    st.session_state.messages = []
if "pending_question" not in st.session_state:
    st.session_state.pending_question = None

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

prompt = st.chat_input("Pregunta sobre los manglares de Greater Guayaquil...")

if st.session_state.pending_question:
    prompt = st.session_state.pending_question
    st.session_state.pending_question = None

if prompt:
    with st.chat_message("user"):
        st.markdown(prompt)
    st.session_state.messages.append({"role": "user", "content": prompt})

    with st.chat_message("assistant"):
        with st.spinner("Consultando..."):
            try:
                client = Anthropic(api_key=ANTHROPIC_API_KEY)
                api_messages = [{"role": m["role"], "content": m["content"]}
                               for m in st.session_state.messages]

                response = client.messages.create(
                    model="claude-sonnet-4-20250514",
                    max_tokens=1024,
                    system=SYSTEM_PROMPT,
                    messages=api_messages
                )

                reply = response.content[0].text
                st.markdown(reply)
                st.session_state.messages.append({"role": "assistant", "content": reply})

            except Exception as e:
                st.error(f"Error: {str(e)}")
                if "AQUI_TU_API_KEY" in ANTHROPIC_API_KEY:
                    st.warning("Reemplaza AQUI_TU_API_KEY con tu API key de Anthropic.")

Overwriting app_manglar.py


In [7]:
!pip install pyngrok

In [8]:
from pyngrok import ngrok
ngrok.set_auth_token("3BaoF1RWX2Nu4xUC9QESNE7kT3p_4urSK9A1jFrSP4qCNSaCe")

In [9]:
!streamlit run app_manglar.py --server.port 8501 --server.headless true &




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.240.222.35:8501

  Stopping...


In [11]:
public_url = ngrok.connect(8501)
print(public_url)

ERROR:pyngrok.process.ngrok:t=2026-03-28T22:38:05+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: The authtoken you specified is properly formed, but it is invalid.\nYour authtoken: 3BaoF1RWX2Nu4xUC9QESNE7kT3p_4urSK9A1jFrSP4qCNSaCe\nThis usually happens when:\n    - You reset your authtoken\n    - Your authtoken was for a team account that you were removed from\n    - You are using ngrok link and this credential was explicitly revoked\nGo to your ngrok dashboard and double check that your authtoken is correct:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_107\r\n"
ERROR:pyngrok.process.ngrok:t=2026-03-28T22:38:05+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: The authtoken you specified is properly formed, but it is invalid.\nYour authtoken: 3BaoF1RWX2Nu4xUC9QESNE7kT3p_4urSK9A1jFrSP4qCNSaCe\nThis usually happens when:\n    - You reset your authtoken\n    - Your authtoken was 

PyngrokNgrokError: The ngrok process errored on start: authentication failed: The authtoken you specified is properly formed, but it is invalid.\nYour authtoken: 3BaoF1RWX2Nu4xUC9QESNE7kT3p_4urSK9A1jFrSP4qCNSaCe\nThis usually happens when:\n    - You reset your authtoken\n    - Your authtoken was for a team account that you were removed from\n    - You are using ngrok link and this credential was explicitly revoked\nGo to your ngrok dashboard and double check that your authtoken is correct:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_107\r\n.